# Baseline Anomaly Detection

In this notebook, we train baseline anomaly detection models using the preprocessed CICIDS2017 network-flow data.

The model is trained only on the Monday BENIGN traffic, which represents normal network behavior. It is then evaluated on the Friday afternoon DDoS file, which contains both BENIGN and DDoS traffic.

The goal is to check whether attack flows receive higher anomaly scores than normal flows.

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

print("Project root:", Path.cwd())

Project root: D:\Development\Python Projects\network-traffic-anomaly-detection


In [2]:
import pandas as pd
import numpy as np

In [3]:
processed_dir = Path("data/processed")

X_train = pd.read_csv(processed_dir / "X_train_scaled.csv")
X_test = pd.read_csv(processed_dir / "X_test_scaled.csv")

y_test = pd.read_csv(processed_dir / "y_test_binary.csv").squeeze()
y_test_labels = pd.read_csv(processed_dir / "y_test_labels.csv").squeeze()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nTest label distribution:")
print(y_test.value_counts())

X_train shape: (502650, 78)
X_test shape: (223082, 78)
y_test shape: (223082,)

Test label distribution:
Label
1    128014
0     95068
Name: count, dtype: int64


## Isolation Forest Baseline

Isolation Forest is used as the first anomaly detection baseline. It works by randomly partitioning the feature space. Anomalous samples are expected to be easier to isolate than normal samples, so they tend to have shorter average path lengths in the isolation trees.

The model is trained only on normal BENIGN traffic. During evaluation, it assigns anomaly scores to both BENIGN and DDoS samples in the test set.

In [5]:
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_train)



IsolationForest(n_jobs=-1, random_state=42)

In [6]:
iso_scores = -iso_forest.decision_function(X_test)

print("Anomaly scores shape:", iso_scores.shape)
print("Min score:", iso_scores.min())
print("Max score:", iso_scores.max())
print("Mean score:", iso_scores.mean())

Anomaly scores shape: (223082,)
Min score: -0.17867838471693676
Max score: 0.27267371440192634
Mean score: -0.031333380132419646


In [7]:
from sklearn.metrics import roc_auc_score, average_precision_score

roc_auc = roc_auc_score(y_test, iso_scores)
pr_auc = average_precision_score(y_test, iso_scores)

print("Isolation Forest ROC-AUC:", roc_auc)
print("Isolation Forest PR-AUC:", pr_auc)

Isolation Forest ROC-AUC: 0.6808983013346402
Isolation Forest PR-AUC: 0.6495483878417078


In [8]:
score_df = pd.DataFrame({
    "label": y_test_labels,
    "binary_label": y_test,
    "anomaly_score": iso_scores
})

score_df.groupby("label")["anomaly_score"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
BENIGN,95068.0,-0.066286,0.118334,-0.178678,-0.165348,-0.112377,-0.019246,0.272674
DDoS,128014.0,-0.005376,0.102728,-0.165953,-0.115220,0.023736,0.036469,0.186137


## Isolation Forest Result Interpretation

The Isolation Forest baseline achieved a ROC-AUC of approximately 0.68 and a PR-AUC of approximately 0.65. This indicates that the model performs better than random guessing, but the separation between BENIGN and DDoS traffic is only moderate.

The anomaly score distribution shows that DDoS samples receive higher anomaly scores on average than BENIGN samples. This suggests that the model has learned some characteristics of normal Monday traffic and can identify part of the DDoS traffic as abnormal.

However, the score ranges of BENIGN and DDoS traffic overlap. Some BENIGN flows are assigned high anomaly scores, while some DDoS flows are assigned lower anomaly scores. This overlap limits the detection performance of the Isolation Forest baseline.

Overall, Isolation Forest provides a useful first baseline, but further models and feature analysis are needed to improve anomaly detection performance.

## Threshold based evaluation

In [9]:
# Compute anomaly scores for training data
train_scores = -iso_forest.decision_function(X_train)

# Set threshold at 95th percentile of training scores
threshold = np.percentile(train_scores, 95)

print("Threshold:", threshold)

Threshold: 0.0494141663787212


In [10]:
# Predict anomaly if score is above threshold
y_pred = (iso_scores > threshold).astype(int)

print(pd.Series(y_pred).value_counts())

0    179787
1     43295
Name: count, dtype: int64


In [11]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["BENIGN", "DDoS"]))

Confusion Matrix:
[[ 74751  20317]
 [105036  22978]]

Classification Report:
              precision    recall  f1-score   support

      BENIGN       0.42      0.79      0.54     95068
        DDoS       0.53      0.18      0.27    128014

    accuracy                           0.44    223082
   macro avg       0.47      0.48      0.41    223082
weighted avg       0.48      0.44      0.39    223082



In [12]:
tn, fp, fn, tp = cm.ravel()

false_positive_rate = fp / (fp + tn)
detection_rate = tp / (tp + fn)

print("False Positive Rate:", false_positive_rate)
print("Detection Rate / Recall:", detection_rate)

False Positive Rate: 0.2137101863928977
Detection Rate / Recall: 0.17949599262580654


## Threshold-Based Isolation Forest Evaluation

Using the 95th percentile of the training anomaly scores as the decision threshold, the Isolation Forest model achieved a low DDoS recall of approximately 0.18. This means that only about 18% of DDoS flows were detected as anomalies.

The false positive rate was approximately 0.21, meaning that around 21% of BENIGN test flows were incorrectly flagged as anomalous.

The confusion matrix shows that the main weakness of this threshold is the high number of false negatives. Many DDoS samples receive anomaly scores below the selected threshold and are therefore classified as normal.

This suggests that although Isolation Forest provides some ranking separation between BENIGN and DDoS traffic, the 95th percentile threshold is too conservative for this experiment. Further threshold analysis is required to study the trade-off between detection rate and false positive rate.

In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for percentile in [50, 60, 70, 80, 85, 90, 95, 97, 99]:
    threshold = np.percentile(train_scores, percentile)
    y_pred = (iso_scores > threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    fpr = fp / (fp + tn)

    threshold_results.append({
        "percentile": percentile,
        "threshold": threshold,
        "precision": precision,
        "recall_detection_rate": recall,
        "f1_score": f1,
        "false_positive_rate": fpr,
        "predicted_anomalies": y_pred.sum()
    })

threshold_results_df = pd.DataFrame(threshold_results)
threshold_results_df

,percentile,threshold,precision,recall_detection_rate,f1_score,false_positive_rate,predicted_anomalies
0,50,-0.145327,0.683457,0.974909,0.803572,0.608007,182604
1,60,-0.132860,0.672574,0.854133,0.752558,0.559915,162571
2,70,-0.114570,0.660326,0.732959,0.694749,0.507700,142095
3,80,-0.085588,0.651059,0.640211,0.645590,0.462038,125881
4,85,-0.060577,0.696089,0.636368,0.664890,0.374122,117031
5,90,-0.027680,0.770613,0.636071,0.696908,0.254954,105664
6,95,0.049414,0.530731,0.179496,0.268264,0.213710,43295
7,97,0.079955,0.654104,0.168255,0.267660,0.119809,32929
8,99,0.132160,0.692262,0.164443,0.265757,0.098435,30409


## Threshold Comparison Interpretation

The threshold comparison shows a clear trade-off between detection rate and false positive rate. Lower thresholds detect more DDoS flows, but they also incorrectly flag a large portion of BENIGN traffic as anomalous. Higher thresholds reduce false positives but miss many DDoS samples.

The 50th percentile threshold achieves very high DDoS recall, but it also produces a false positive rate of approximately 61%, which would be too noisy for a practical monitoring system.

The 90th percentile threshold provides a more balanced operating point. It achieves approximately 77% precision, 64% recall, and an F1-score of around 0.70, while keeping the false positive rate lower than the more aggressive thresholds.

Therefore, the 90th percentile threshold is selected as the practical Isolation Forest baseline threshold for this experiment.

In [14]:
selected_percentile = 90
selected_threshold = np.percentile(train_scores, selected_percentile)

y_pred_iso = (iso_scores > selected_threshold).astype(int)

print("Selected percentile:", selected_percentile)
print("Selected threshold:", selected_threshold)
print("Predicted anomalies:", y_pred_iso.sum())

Selected percentile: 90
Selected threshold: -0.027679601296038792
Predicted anomalies: 105664


In [15]:
from sklearn.metrics import confusion_matrix, classification_report

cm_iso = confusion_matrix(y_test, y_pred_iso)

print("Confusion Matrix:")
print(cm_iso)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_iso, target_names=["BENIGN", "DDoS"]))

Confusion Matrix:
[[70830 24238]
 [46588 81426]]

Classification Report:
              precision    recall  f1-score   support

      BENIGN       0.60      0.75      0.67     95068
        DDoS       0.77      0.64      0.70    128014

    accuracy                           0.68    223082
   macro avg       0.69      0.69      0.68    223082
weighted avg       0.70      0.68      0.68    223082



In [16]:
tn, fp, fn, tp = cm_iso.ravel()

fpr_iso = fp / (fp + tn)
recall_iso = tp / (tp + fn)

print("False Positive Rate:", fpr_iso)
print("Detection Rate / Recall:", recall_iso)

False Positive Rate: 0.25495434846636095
Detection Rate / Recall: 0.6360710547283891


In [17]:
iso_summary = pd.DataFrame([{
    "model": "Isolation Forest",
    "threshold_percentile": selected_percentile,
    "roc_auc": roc_auc,
    "pr_auc": pr_auc,
    "precision": 0.770613,
    "recall": 0.636071,
    "f1_score": 0.696908,
    "false_positive_rate": 0.254954
}])

iso_summary

,model,threshold_percentile,roc_auc,pr_auc,precision,recall,f1_score,false_positive_rate
0,Isolation Forest,90,0.680898,0.649548,0.770613,0.636071,0.696908,0.254954


## Isolation Forest Baseline Summary

Isolation Forest was trained only on the Monday BENIGN traffic and evaluated on the Friday afternoon BENIGN + DDoS traffic. The model achieved a ROC-AUC of approximately 0.68 and a PR-AUC of approximately 0.65, showing moderate ability to rank DDoS flows as more anomalous than BENIGN flows.

A threshold comparison was performed using different percentiles of the training anomaly score distribution. Lower thresholds increased DDoS detection but caused high false positive rates. Higher thresholds reduced false positives but missed many attacks.

The 90th percentile threshold was selected as the most practical operating point. At this threshold, the model achieved approximately 77% precision, 64% recall, and an F1-score of about 0.70, with a false positive rate of about 25%.

This result provides a useful first baseline, but the false positive rate is still high for a real monitoring system. Therefore, additional anomaly detection methods should be evaluated and compared.